# B5.2 — Pre-test validation with the frozen module (`candidates_v2`, commit frozen in B5.1)

No decisions are made here: independent checks of the frozen pipelines before any test split is touched. All counts come from `src/pipeline/candidates_v2.py` (shared core), so this notebook does not depend on the pipeline wrapper names.

| Part | Data | What | Label |
|---|---|---|---|
| F | FSC-147 val (held-out 1,186 + dev100) | single-class mode **with the ladder** on every image whose v1 mask set was empty (per system); v1 → v1.1 metrics | descriptive, post-hoc (B4c selection unchanged) |
| V | OmniCount-191 val held-out: val clean minus the 262 B5.0 design images; 80 per available domain (seed 0) | multi mode (U_both), both systems, via `run_batch` exactly as in B6; both metrics; comparison with Part 4 (train) on the same domains; B6 timing estimates | validation |

Run on an **L4** runtime: Run all (resumable — after a disconnect, Run all again; finished images are skipped).

In [ ]:
# Step 0.1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# Step 0.2 — Paths, installs (SAN + detectron2 + diffusers, protobuf last), repo
# ============================================================
import importlib, json, re, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd
import torch
assert torch.cuda.is_available(), "Use a GPU runtime (L4)."
PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
B50 = RESULTS_ROOT / "B5_0_OmniCount_Region_Tests"
OUT_DIR = RESULTS_ROOT / "B5_2_PreTest_Validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OMNICOUNT_ROOT = PROJECT_ROOT / "datasets" / "OmniCount-191"
FSC147_IMAGE_DIR = PROJECT_ROOT / "datasets" / "FSC147" / "raw" / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| L4 expected for exact live checks")

DETECTRON2_COMMIT = "1e3e13bbf607b54f62205c4c33922521822fb298"
%cd /content
if not Path("/content/SAN").exists():
    !git clone -q https://github.com/MendelXu/SAN.git /content/SAN
!pip install -q cython scipy shapely timm h5py submitit scikit-image wandb setuptools fvcore tabulate regex opencv-python "open_clip_torch==2.16.0"
!pip uninstall -y -q detectron2
!python -m pip install -q --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git@{DETECTRON2_COMMIT}"
!pip install -q "accelerate==1.14.0" "diffusers==0.40.0" "transformers==5.16.1"
!pip install -q "protobuf==3.20.3" "iopath==0.1.10"


REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
for p in [str(REPO_DIR), "/content/SAN"]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Repo:", REPO_COMMIT[:10])

GPU: NVIDIA L4 | L4 expected for exact live checks
/content
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wandb 0.25.0 requires protobuf!=4.21.0,!=5.28.0,<7,>=3.19.0; python_version > "3.9" and sys_platform == "linux", but you have protobuf 7.36.2 which is incompatible.
open-clip-torch 2.16.0 requires protobuf<4, but you have protobuf 7.36.2 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.36.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
detectron2 0.6 requires iopath<0.1.10,>=0.1.7, but you have iopath 0.1.10 which is incompatible.
google-cloud-resource-manager 1.19.0 requ

In [ ]:
# ============================================================
# Step 0.3 — SAM2 (frozen commit), module imports, models
# ============================================================
import google.protobuf, importlib.metadata as md_meta
assert md_meta.version("protobuf") == "3.20.3" and google.protobuf.__version__ == "3.20.3", \
    "protobuf mismatch — restart the session and Run all again."
import src.pipeline.b0 as b0
%cd /content
if not Path("/content/sam2_repo/sam2/__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")
importlib.invalidate_caches()

import src.datasets as datasets_mod
import src.pipeline.semantic as semantic
import src.pipeline.prompts as prompts
import src.pipeline.candidates as C1
import src.pipeline.candidates_v2 as core

SAN_ROOT = Path("/content/SAN"); SAN_CKPT = SAN_ROOT / "output" / "san_vit_b_16.pth"
SAN_CKPT.parent.mkdir(parents=True, exist_ok=True)
if not SAN_CKPT.exists():
    shutil.copy2(PROJECT_ROOT / "Checkpoints" / "SAN" / "san_vit_b_16.pth", SAN_CKPT)
SAM2_CKPT = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_large.pt"
M = core.load_models(SAN_ROOT, SAN_CKPT, SAM2_CKPT, systems=("A", "B"), device="cuda")
frz = json.load(open(RESULTS_ROOT / "B5_1_Candidates_v2_Freeze" / "B5_1_freeze_record.json"))
assert frz["all_pass"], "B5.1 freeze record does not show all checks passing"
print("B5.1 freeze: commit", frz["repo_commit"][:10], "| current repo:", REPO_COMMIT[:10])
print(core.VERSION, "| models loaded:", "SAN, SAM2 predictor,", sorted(M.amg), "| normals:", M.normals is not None)
import src.evaluation as ev
from tqdm.auto import tqdm
from PIL import Image

/content


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
100%|███████████████████████████████████████| 351M/351M [00:07<00:00, 48.4MiB/s]
/content/SAN/san/model/side_adapter/timm_wrapper.py:24: UserWarning: Unused kwargs are provided:{'dynamic_img_pad': False, 'device': None, 'dtype': None}.
  warnings.warn(f"Unused kwargs are provided:{kwargs}.")


model_index.json:   0%|          | 0.00/512 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

B5.1 freeze: commit 2086087c5e | current repo: 2086087c5e
candidates_v2 | models loaded: SAN, SAM2 predictor, ['LM0', 'MixA'] | normals: True


## Part F — FSC-147 val: ladder on empty-mask-set images (v1 → v1.1, descriptive)

In [ ]:
# ============================================================
# Step F.1 — Images whose v1 mask set was empty (per system), single-class mode with the ladder
# ============================================================
B4C = RESULTS_ROOT / "B4c_HeldOut_Candidates"
full = pd.read_csv(B4C / "B4c_fullval_per_image.csv")
ho_ids = set(pd.read_csv(B4C / "B4c_heldout_per_image.csv")["dataset_index"])
full["subset"] = np.where(full["dataset_index"].isin(ho_ids), "held-out", "dev100")
COL = {"A": "GPV_R4+P[MixA]", "B": "GPV_K-n+P[LM0]"}; BASE = {"A": "MixA", "B": "LM0"}
emp = full[(full["n_r4[MixA]"] == 0) | (full["n_r4[LM0]"] == 0)].reset_index(drop=True)
print(f"Images with an empty v1 mask set: {len(emp)} (held-out {int((emp['subset'] == 'held-out').sum())}, "
      f"dev100 {int((emp['subset'] == 'dev100').sum())}) | system A empty: {int((emp['n_r4[MixA]'] == 0).sum())}, "
      f"system B empty: {int((emp['n_r4[LM0]'] == 0).sum())}")
F_ROWS = OUT_DIR / "fsc_ladder_rows"; F_ROWS.mkdir(parents=True, exist_ok=True)
for r in tqdm(list(emp.itertuples(index=False)), desc="FSC ladder"):
    fp = F_ROWS / f"{int(r.dataset_index):05d}.json"
    if fp.exists():
        continue
    out = core.count_image(M, FSC147_IMAGE_DIR / r.image_id, [r.category], "single", systems=("A", "B"), ladder=True)
    fp.write_text(json.dumps({"dataset_index": int(r.dataset_index), "counts": {s: out["counts"][s][r.category] for s in "AB"},
                              "levels": {s: out["diag"][f"{s}_ladder_level"] for s in "AB"}}))
lad = pd.DataFrame([json.load(open(p)) for p in sorted(F_ROWS.glob("*.json"))])
for s in "AB":
    lad[f"{s}_v1.1"] = lad["counts"].apply(lambda d: d[s]); lad[f"{s}_level"] = lad["levels"].apply(lambda d: d[s])
emp = emp.merge(lad[["dataset_index", "A_v1.1", "A_level", "B_v1.1", "B_level"]], on="dataset_index", validate="one_to_one")
# Where a system's v1 set was NOT empty, the live count should equal v1 (held-out caches were built on L4 in B4b)
for s in "AB":
    nonempty = emp[f"n_r4[{BASE[s]}]"] > 0
    agree = (emp.loc[nonempty, f"{s}_v1.1"] == emp.loc[nonempty, COL[s]])
    print(f"System {s}: v1 set non-empty on {int(nonempty.sum())} of these images; live == v1 on {int(agree.sum())}/{int(nonempty.sum())}")
display(emp[["subset", "image_id", "category", "gt_count", "n_r4[MixA]", COL["A"], "A_v1.1", "A_level",
             "n_r4[LM0]", COL["B"], "B_v1.1", "B_level"]])

Images with an empty v1 mask set: 22 (held-out 20, dev100 2) | system A empty: 22, system B empty: 22


FSC ladder:   0%|          | 0/22 [00:00<?, ?it/s]

System A: v1 set non-empty on 0 of these images; live == v1 on 0/0
System B: v1 set non-empty on 0 of these images; live == v1 on 0/0


,subset,image_id,category,gt_count,n_r4[MixA],GPV_R4+P[MixA],A_v1.1,A_level,n_r4[LM0],GPV_K-n+P[LM0],B_v1.1,B_level
0,held-out,2858.jpg,chairs,131,0,0,57,1,0,0,467,1
1,held-out,3428.jpg,polka dots,458,0,0,11,2,0,0,13,2
2,held-out,3486.jpg,polka dots,81,0,0,4,2,0,0,4,2
3,held-out,4855.jpg,grapes,22,0,0,31,1,0,0,26,1
4,held-out,5044.jpg,polka dots,100,0,0,5,2,0,0,7,2
5,held-out,5151.jpg,toilet paper rolls,32,0,0,1,1,0,0,1,1
6,held-out,5742.jpg,polka dots,43,0,0,9,2,0,0,7,2
7,held-out,5746.jpg,polka dots,100,0,0,5,2,0,0,7,2
8,held-out,5748.jpg,polka dots,74,0,0,9,1,0,0,8,1
9,held-out,7532.jpg,fresh cut,20,0,0,28,2,0,0,27,2


In [ ]:
# ============================================================
# Step F.2 — v1 vs v1.1 metrics (counts replaced only where that system's v1 mask set was empty)
# ============================================================
full11 = full.copy()
for s in "AB":
    m = emp.set_index("dataset_index")
    sel = full11["dataset_index"].isin(m.index[m[f"n_r4[{BASE[s]}]"] == 0])
    full11[f"{COL[s]} v1.1"] = full11[COL[s]]
    full11.loc[sel, f"{COL[s]} v1.1"] = full11.loc[sel, "dataset_index"].map(m[f"{s}_v1.1"]).to_numpy()
STRATA = sorted(full["density_stratum"].unique())
def metrics(df, col):
    e = df[col] - df["gt_count"]; ae = e.abs(); k = max(1, int(round(0.01 * len(df))))
    row = {"MAE": ae.mean(), "RMSE": float(np.sqrt((e ** 2).mean())), "trimmed_MAE": np.sort(ae)[:len(ae) - k].mean(),
           "NAE": (ae / df["gt_count"]).mean()}
    row.update({f"MAE_{x[:2]}": ae[df["density_stratum"] == x].mean() for x in STRATA})
    return row
rows = []
for label, df in [("held-out (1,186)", full11[full11["subset"] == "held-out"]), ("full val (1,286)", full11)]:
    for s in "AB":
        for ver, col in [("v1 (B4c)", COL[s]), ("v1.1 (+ ladder)", f"{COL[s]} v1.1")]:
            rows.append({"set": label, "system": COL[s], "version": ver, **metrics(df, col)})
fsc_m = pd.DataFrame(rows)
print("FSC-147 val — v1 vs v1.1 (descriptive, post-hoc; B4c selection unchanged):"); display(fsc_m.round(2))
full11.to_csv(OUT_DIR / "B5_2_fsc_val_v1_1_per_image.csv", index=False)
fsc_m.to_csv(OUT_DIR / "B5_2_fsc_val_v1_vs_v1_1.csv", index=False)

FSC-147 val — v1 vs v1.1 (descriptive, post-hoc; B4c selection unchanged):


,set,system,version,MAE,RMSE,trimmed_MAE,NAE,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,"held-out (1,186)",GPV_R4+P[MixA],v1 (B4c),27.07,88.33,20.23,0.42,6.74,7.00,8.86,14.76,41.10,286.53
1,"held-out (1,186)",GPV_R4+P[MixA],v1.1 (+ ladder),26.38,87.75,19.55,0.41,6.74,6.95,8.79,13.91,37.96,286.33
2,"held-out (1,186)",GPV_K-n+P[LM0],v1 (B4c),28.43,85.73,22.35,0.38,4.50,7.13,7.71,16.24,48.53,299.07
3,"held-out (1,186)",GPV_K-n+P[LM0],v1.1 (+ ladder),28.00,85.67,21.93,0.38,4.50,7.07,7.59,15.27,47.30,298.84
4,"full val (1,286)",GPV_R4+P[MixA],v1 (B4c),30.01,110.55,21.33,0.41,6.57,7.01,8.89,14.51,40.40,328.20
5,"full val (1,286)",GPV_R4+P[MixA],v1.1 (+ ladder),29.33,109.76,20.69,0.41,6.57,6.96,8.82,13.72,37.40,327.15
6,"full val (1,286)",GPV_K-n+P[LM0],v1 (B4c),31.31,108.70,23.26,0.38,4.59,6.86,7.78,16.04,47.76,338.92
7,"full val (1,286)",GPV_K-n+P[LM0],v1.1 (+ ladder),30.91,108.65,22.87,0.38,4.59,6.81,7.67,15.14,46.58,338.71


## Part V — OmniCount-191 val held-out (untouched images), multi mode via `run_batch`

In [ ]:
# ============================================================
# Step V.1 — Held-out pool: val clean minus all B5.0 design images; 80 per available domain (seed 0)
# ============================================================
B50 = RESULTS_ROOT / "B5_0_OmniCount_Region_Tests"
clean = pd.read_csv(B50 / "part1_inventory" / "B5_0_val_clean_images.csv")
clean["gt"] = clean["gt"].apply(json.loads)
design = set(pd.read_csv(B50 / "part2_semantic" / "B5_0_p2_queries.csv")["image_id"])
assert len(design) == 262, len(design)
pool = clean[~clean["image_id"].isin(design)]
N_V = 80
vsub = pd.concat([g.sample(n=min(N_V, len(g)), random_state=0) for _, g in pool.groupby("domain")], ignore_index=True)
val_ds = datasets_mod.OmniCount191Dataset(OMNICOUNT_ROOT, split="val", load_images=False)
VOCAB = val_ds.domain_vocabulary
inv = pd.DataFrame({"val_clean": clean.groupby("domain").size(), "design_used": clean[clean["image_id"].isin(design)].groupby("domain").size(),
                    "pool": pool.groupby("domain").size(), "sampled": vsub.groupby("domain").size()}).fillna(0).astype(int)
display(inv)
print(f"Held-out subset: {len(vsub)} images | present queries {int(vsub['gt'].apply(len).sum())} | "
      f"standard queries {sum(len(VOCAB[d]) for d in vsub['domain'])}")
vsub.assign(gt=vsub["gt"].apply(json.dumps)).to_csv(OUT_DIR / "B5_2_omnicount_val_heldout_subset.csv", index=False)

,val_clean,design_used,pool,sampled
domain,,,,
Birds,5,5,0,0
Fruits,256,40,216,80
Pets,15,15,0,0
Satellite,254,40,214,80
Supermarket,477,40,437,80
Urban,2000,80,1920,80
Vegetables,2,2,0,0
Wild,505,40,465,80


Held-out subset: 400 images | present queries 731 | standard queries 7920


In [7]:
# ============================================================
# Step V.2 — run_batch (multi mode, both systems) — resumable, raw predictions saved per image
# ============================================================
V_DIR = OUT_DIR / "omnicount_val_heldout"
items = [{"key": r.image_id.replace("/", "__"), "image_path": r.image_path, "classes": VOCAB[r.domain],
          "prompts": {c: datasets_mod.omnicount_prompt_name(c) for c in VOCAB[r.domain]},
          "meta": {"domain": r.domain, "idx": int(r.idx), "image_id": r.image_id}} for r in vsub.itertuples(index=False)]
t0 = time.time()
n_new = core.run_batch(M, items, V_DIR / "rows", "multi", systems=("A", "B"))
print(f"Processed now: {n_new} | elapsed {(time.time() - t0) / 60:.1f} min")

candidates_v2 multi:   0%|          | 0/400 [00:00<?, ?it/s]

Processed now: 400 | elapsed 149.2 min


In [8]:
# ============================================================
# Step V.3 — Evaluation (both metrics) and comparison with Part 4 (train) on the same domains
# ============================================================
LOCAL = Path("/content/B5_2_rows_local")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
shutil.copytree(V_DIR / "rows", LOCAL)
recs = [json.load(open(LOCAL / f"{it['key']}.json")) for it in items]
SYS = {"A": "GPV_R4+P[MixA]", "B": "GPV_K-n+P[LM0]"}

def results_from(recs, s, domains=None):
    out = []
    for rec in recs:
        d = rec["meta"]["domain"]
        if domains is not None and d not in domains:
            continue
        out.append({"image_id": rec["meta"]["image_id"], "domain": d, "categories": val_ds[rec["meta"]["idx"]]["categories"],
                    "pred_counts": rec["counts"][s], "query_classes": VOCAB[d]})
    return out

def summarise(res):
    pairs = ev.build_multi_label_pairs(res); _, _, m = ev.evaluate_multi_label_counts(pairs); _, om = ev.omnicount_reference_by_domain(res)
    return {"MAE_nz": m["MAE_nz"], "RMSE_nz": m["RMSE_nz"], "NAE_nz": m["NAE_nz"], "std mRMSE": m["mRMSE"],
            "std mRMSE_nz": m["mRMSE_nz"], "FP units": m["false_positive_units"],
            "OmniCount m_rmse": om["pooled_m_rmse"], "OmniCount m_rmse_nz": om["pooled_m_rmse_nz"]}

DOMS = sorted(vsub["domain"].unique()); NO_WILD = [d for d in DOMS if d != "Wild"]
val_rows = [{"system": SYS[s], "set": lab, **summarise(results_from(recs, s, doms))}
            for s in "AB" for lab, doms in [("all domains", DOMS), ("without Wild", NO_WILD)]]
vm = pd.DataFrame(val_rows)
print("OmniCount-191 val held-out (both metrics):"); display(vm.round(3))

# Part 4 (train) on the same domains, U_both
e4 = pd.read_csv(B50 / "part4_train_confirmation" / "B5_0_p4_counts.csv")
e4 = e4[e4["domain"].isin(DOMS)]
p4_rows = []
for s in "AB":
    col = f"{SYS[s]}|U_both"; res = []
    for iid, g in e4.groupby("image_id"):
        res.append({"image_id": iid, "domain": g["domain"].iloc[0],
                    "categories": {c: {"count": int(n)} for c, n in zip(g["class"], g["gt_count"]) if n > 0},
                    "pred_counts": dict(zip(g["class"], g[col])), "query_classes": list(g["class"])})
    p4_rows.append({"system": SYS[s], "set": "Part 4 train (same domains)", **summarise(res)})
cmp_ = pd.concat([vm[vm["set"] == "all domains"].assign(set="val held-out (this)"), pd.DataFrame(p4_rows)], ignore_index=True)
print("\nConsistency: val held-out vs Part 4 train (same 5 domains, U_both):"); display(cmp_.round(3))

per_dom = []
for s in "AB":
    for d in DOMS:
        r_ = summarise(results_from(recs, s, [d]))
        per_dom.append({"system": SYS[s], "domain": d, "MAE_nz": r_["MAE_nz"], "std mRMSE": r_["std mRMSE"], "FP units": r_["FP units"]})
pdm = pd.DataFrame(per_dom).pivot(index="domain", columns="system"); print("\nPer domain:"); display(pdm.round(2))
pairsA = ev.build_multi_label_pairs(results_from(recs, "A")); pairsB = ev.build_multi_label_pairs(results_from(recs, "B"))
for nm, pr in [("A", pairsA), ("B", pairsB)]:
    pr = pd.DataFrame(pr); P = pr[pr["gt_count"] > 0]; Ab = pr[pr["gt_count"] == 0]
    print(f"System {nm}: present counted 0: {int((P['pred_count'] == 0).sum())}/{len(P)} | absent counted > 0: {int((Ab['pred_count'] > 0).sum())}/{len(Ab)}")

OmniCount-191 val held-out (both metrics):


,system,set,MAE_nz,RMSE_nz,NAE_nz,std mRMSE,std mRMSE_nz,FP units,OmniCount m_rmse,OmniCount m_rmse_nz
0,GPV_R4+P[MixA],all domains,4.330,14.258,2.009,2.473,9.457,2760.0,0.180,1.613
1,GPV_R4+P[MixA],without Wild,4.666,15.094,2.179,2.465,9.486,2423.0,0.248,2.009
2,GPV_K-n+P[LM0],all domains,5.185,19.413,2.387,3.059,10.331,2652.0,0.236,1.763
3,GPV_K-n+P[LM0],without Wild,5.242,19.335,2.276,2.868,10.104,2277.0,0.298,2.142



Consistency: val held-out vs Part 4 train (same 5 domains, U_both):


,system,set,MAE_nz,RMSE_nz,NAE_nz,std mRMSE,std mRMSE_nz,FP units,OmniCount m_rmse,OmniCount m_rmse_nz
0,GPV_R4+P[MixA],val held-out (this),4.330,14.258,2.009,2.473,9.457,2760.0,0.180,1.613
1,GPV_K-n+P[LM0],val held-out (this),5.185,19.413,2.387,3.059,10.331,2652.0,0.236,1.763
2,GPV_R4+P[MixA],Part 4 train (same domains),2.821,7.889,1.153,1.457,4.303,1365.0,0.142,0.958
3,GPV_K-n+P[LM0],Part 4 train (same domains),3.571,10.871,1.537,1.874,5.509,1412.0,0.195,1.231



Per domain:


MAE_nz                     std mRMSE                 \
system      GPV_K-n+P[LM0] GPV_R4+P[MixA] GPV_K-n+P[LM0] GPV_R4+P[MixA]   
domain                                                                    
Fruits                0.50           0.58           0.71           0.75   
Satellite            18.53          17.80           5.65           4.51   
Supermarket          10.91          11.42           1.81           2.08   
Urban                 4.60           1.99           3.56           1.38   
Wild                  4.73           1.63           7.58           2.65   

                  FP units                 
system      GPV_K-n+P[LM0] GPV_R4+P[MixA]  
domain                                     
Fruits                55.0           62.0  
Satellite           1050.0          899.0  
Supermarket          909.0         1117.0  
Urban                263.0          345.0  
Wild                 375.0          337.0

System A: present counted 0: 204/731 | absent counted > 0: 558/7189
System B: present counted 0: 210/731 | absent counted > 0: 509/7189


In [9]:
# ============================================================
# Step V.4 — Timing and B6 estimates; save
# ============================================================
tt = pd.DataFrame([{"domain": r["meta"]["domain"], **{k: v for k, v in r["diag"].items() if k.startswith("t_")}} for r in recs])
print("Mean time per image (s), both systems in one run:"); display(tt.groupby("domain").mean().round(1))
per_img = tt["t_total_s"].mean()
print(f"\nOmniCount multi mode: {per_img:.1f} s/image (both systems) -> OmniCount test copy (1,957 images): "
      f"~{1957 * per_img / 3600:.1f} h")
vm.to_csv(OUT_DIR / "B5_2_omnicount_val_metrics.csv", index=False)
cmp_.to_csv(OUT_DIR / "B5_2_omnicount_val_vs_part4.csv", index=False)
pdm.to_csv(OUT_DIR / "B5_2_omnicount_val_per_domain.csv")
tt.to_csv(OUT_DIR / "B5_2_omnicount_val_timing.csv", index=False)
with open(OUT_DIR / "B5_2_metadata.json", "w") as f:
    json.dump({"notebook": "B5_2_PreTest_Validation", "module": core.VERSION, "repo_commit": REPO_COMMIT, "gpu": GPU_NAME,
               "freeze_commit": frz["repo_commit"], "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "fsc_ladder_images": len(emp), "omnicount_val_images": len(vsub), "test_split_used": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Mean time per image (s), both systems in one run:


,t_evidence_s,t_normals_s,t_A_s,t_B_s,t_total_s
domain,,,,,
Fruits,1.3,0.7,16.7,5.0,23.7
Satellite,1.2,0.7,13.4,3.6,18.8
Supermarket,2.7,0.7,16.5,5.1,25.0
Urban,1.6,0.7,15.7,4.5,22.5
Wild,1.1,0.7,15.6,4.3,21.8



OmniCount multi mode: 22.4 s/image (both systems) -> OmniCount test copy (1,957 images): ~12.2 h
Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B5_2_PreTest_Validation


In [10]:
# Diagnostic (CPU): Satellite / Supermarket — val held-out vs Part 4 train
B50 = RESULTS_ROOT / "B5_0_OmniCount_Region_Tests"; V = OUT_DIR / "omnicount_val_heldout" / "rows"
rows = []
for p in V.glob("*.json"):
    r = json.load(open(p)); gt = {}
    sub_ = pd.read_csv(OUT_DIR / "B5_2_omnicount_val_heldout_subset.csv")
    break
sub_ = pd.read_csv(OUT_DIR / "B5_2_omnicount_val_heldout_subset.csv"); sub_["gt"] = sub_["gt"].apply(json.loads)
gtmap = dict(zip(sub_["image_id"], sub_["gt"]))
for p in V.glob("*.json"):
    r = json.load(open(p)); iid = r["meta"]["image_id"]; d = r["meta"]["domain"]
    for c, g in gtmap[iid].items():
        rows.append({"set": "val", "domain": d, "image_id": iid, "class": c, "gt": g, "A": r["counts"]["A"].get(c, 0)})
e4 = pd.read_csv(B50 / "part4_train_confirmation" / "B5_0_p4_counts.csv")
e4 = e4[e4["present"]].rename(columns={"gt_count": "gt", "GPV_R4+P[MixA]|U_both": "A"})
dv = pd.concat([pd.DataFrame(rows), e4.assign(set="train")[["set", "domain", "image_id", "class", "gt", "A"]]])
dv["ae"] = (dv["A"] - dv["gt"]).abs()
for d in ["Satellite", "Supermarket"]:
    g = dv[dv["domain"] == d]
    print(f"\n{d}: present-class GT counts and primary error, val vs train")
    display(g.groupby("set").agg(n=("gt", "size"), gt_mean=("gt", "mean"), gt_median=("gt", "median"), gt_max=("gt", "max"),
                                 mae=("ae", "mean"), zeros=("A", lambda x: int((x == 0).sum()))).round(2))
    display(g[g["set"] == "val"].groupby("class").agg(n=("gt", "size"), gt_mean=("gt", "mean"), mae=("ae", "mean"))
            .sort_values("mae", ascending=False).head(8).round(1))
    display(g[g["set"] == "val"].sort_values("ae", ascending=False).head(8)[["image_id", "class", "gt", "A"]])


Satellite: present-class GT counts and primary error, val vs train


,n,gt_mean,gt_median,gt_max,mae,zeros
set,,,,,,
train,50,1.36,1.0,8,4.76,29
val,81,2.28,1.0,19,17.80,40


,n,gt_mean,mae
class,,,
parking area,4,4.5,92.8
houses,6,2.8,67.3
School,4,1.0,54.0
Farmlands,8,1.0,27.8
Baseball field,4,1.8,17.0
building,8,6.9,9.5
tank,2,6.0,5.5
bridge,3,1.7,4.3


,image_id,class,gt,A
352,Satellite/val/79,houses,3,136
331,Satellite/val/6,houses,1,133
390,Satellite/val/186,parking area,1,119
374,Satellite/val/123,parking area,4,105
360,Satellite/val/181,parking area,5,99
355,Satellite/val/167,Farmlands,1,73
367,Satellite/val/159,houses,4,74
395,Satellite/val/124,School,1,68



Supermarket: present-class GT counts and primary error, val vs train


,n,gt_mean,gt_median,gt_max,mae,zeros
set,,,,,,
train,60,11.52,3.0,136,9.90,25
val,97,12.38,2.0,68,11.42,39


,n,gt_mean,mae
class,,,
Coffee,14,38.3,38.3
Sauce,1,1.0,36.0
Beans,3,37.3,33.3
Butter,2,24.0,24.0
Drinks,6,39.2,19.5
Brussel Sprouts,1,2.0,12.0
Detergent,4,11.5,11.8
Cantaloupe,2,9.5,9.5


,image_id,class,gt,A
451,Supermarket/val/100,Drinks,68,1
426,Supermarket/val/107,Beans,63,0
468,Supermarket/val/410,Coffee,62,0
486,Supermarket/val/188,Coffee,57,0
441,Supermarket/val/392,Coffee,53,0
472,Supermarket/val/261,Coffee,52,0
477,Supermarket/val/425,Butter,47,0
415,Supermarket/val/352,Coffee,46,0
